<a href="https://colab.research.google.com/github/SebasFru/Terminal-de-Economia./blob/main/Proyecto_Final_Eco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [144]:
import pandas as pd
import numpy as np
import plotly.express as px

Descargamos y visualizamos la base de datos principal, donde se encuentra la informacion de los créditos y sus calificaciones.

In [145]:
df_principal = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Proyecto/040_R04A_417_10.csv')

In [146]:
df_principal.head(3)

,sector,periodo,institucion,moneda,tipo_saldo,orden_presentacion,concepto,importe_pesos
0,40,202501,40002,14,10,10,101800103003,-1.024114e+10
1,40,202501,40002,4,10,10,101800103003,-5.785247e+08
2,40,202501,40002,14,10,20,111800304005,-4.978340e+09


Descargamos y visualizamos la base de datos la información de las intituciones, su clave de institución y grupo.

In [147]:
df_instituciones = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Proyecto/catalogo_instituciones_BM (1).csv", encoding='latin-1')

In [148]:
df_instituciones.head(3)

,clavesector,sector,claveinstitucion,nombreinstitucion,clavegrupo,nombregrupo
0,40,BANCA MÚLTIPLE,40002,Banamex,59.0,G-7
1,40,BANCA MÚLTIPLE,40012,BBVA México,59.0,G-7
2,40,BANCA MÚLTIPLE,40014,Santander,59.0,G-7


En esta otra base de datos se encuentran la descripcion asociada a los conceptos del df_principal.

In [149]:
df_claves = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Proyecto/catalogo_R04A_417_BM (1).csv", encoding='latin-1')

In [150]:
df_claves.head(3)

,concepto,descripcion,descripcion_identada,orden
0,101800103003,Estimaciones totales (I+II+III+IV),Estimaciones totales (I+II+III+IV),10
1,111800304005,I. Cartera base y estimaciones derivadas de la...,I. Cartera base y estimaciones derivadas de l...,20
2,101800505014,A). Cartera de crédito,A). Cartera de crédito,30


Debido a que la base de datos de conceptos no esta identada correctamente, tuvimos que estructurar todo nuevamente, analizando la base de datso tenemos que esta de la sigueinte forma:


[Niveles y Jerarquias](https://drive.google.com/file/d/1oce3Qxbl1X8nBJt3p9emInyJvY4geGi6/view?usp=drive_link)

In [151]:
def reestructurar_catalogo_colab(df):
    def determinar_nivel(descripcion, concepto):
        if pd.isna(descripcion) or descripcion == "":
            return 0

        desc = str(descripcion).strip()
        concepto_str = str(concepto)

        # Nivel 1: Estimaciones totales (raíz)
        if desc == "Estimaciones totales (I+II+III+IV)":
            return 1

        # Nivel 2: Etapas de riesgo (I., II., III., IV.)
        elif desc.startswith(("I. ", "II. ", "III. ", "IV. ")):
            return 2

        # Nivel 3: Tipo de operación (A)., B).)
        elif desc.startswith(("A). ", "B). ")):
            return 3

        # Nivel 4: Clases de crédito (1., 2., 3.)
        elif desc.startswith(("1. ", "2. ", "3. ")) and not desc.startswith("1. Avales") and not desc.startswith("2. Compromisos"):
            return 4

        # Nivel 5: Subclases específicas
        elif any(x in desc for x in [
            "Actividad empresarial", "Entidades financieras", "Entidades gubernamentales",
            "Tarjeta de crédito", "Personales", "Nómina", "Automotriz",
            "Media y residencial", "De interés social", "Créditos adquiridos",
            "Remodelación o mejoramiento"
        ]):
            return 5

        # Nivel 6: Operaciones específicas
        elif any(x in desc for x in [
            "Operaciones quirografarias", "Operaciones prendarias",
            "Operaciones de habilitación o avío", "Operaciones refaccionarias",
            "Operaciones de factoraje financiero", "Operaciones de arrendamiento financiero",
            "Operaciones con garantía hipotecaria", "Créditos puente",
            "Créditos para proyectos de inversión", "Créditos interbancarios",
            "Créditos al gobierno federal", "Créditos a estados y municipios",
            "Créditos a empresas productivas", "Créditos a organismos descentralizados",
            "Adquisición de bienes muebles", "Microcréditos", "Otros créditos de consumo",
            "Cartera ordinaria", "Cartera en prórroga", "Régimen especial de amortización",
            "Fideicomisos públicos de contratación", "Otros"
        ]):
            return 6

        # Nivel 7: Calificaciones de riesgo
        elif desc.startswith("Riesgo "):
            return 7

        # Nivel 8: Piso MI
        elif desc == "Piso MI":
            return 8

        # Nivel 9: Avales y compromisos (subtipos de operaciones fuera de balance)
        elif desc.startswith(("1. Avales", "2. Compromisos")):
            return 9

        # Nivel 10: Exceptuada
        elif desc == "Exceptuada":
            return 10

        else:
            # Por defecto, considerar como nivel 4
            return 4

    # Aplicar determinación de niveles
    df['nivel_corregido'] = df.apply(lambda x: determinar_nivel(x['descripcion'], x['concepto']), axis=1)

    # Inicializar columnas de niveles
    for i in range(1, 12):
        df[f'Nivel_{i}'] = None

    # Variables para rastrear la jerarquía actual
    current_path = {}
    last_nivel_1 = None
    last_nivel_2 = None
    last_nivel_3 = None
    last_nivel_4 = None
    last_nivel_5 = None
    last_nivel_6 = None

    # Reconstruir la jerarquía
    for idx, row in df.iterrows():
        nivel = row['nivel_corregido']
        descripcion = row['descripcion']

        if nivel == 0:
            continue

        # Actualizar el camino según el nivel actual
        if nivel == 1:
            last_nivel_1 = descripcion
            current_path = {1: last_nivel_1}
        elif nivel == 2:
            last_nivel_2 = descripcion
            current_path = {1: last_nivel_1, 2: last_nivel_2}
        elif nivel == 3:
            last_nivel_3 = descripcion
            current_path = {1: last_nivel_1, 2: last_nivel_2, 3: last_nivel_3}
        elif nivel == 4:
            last_nivel_4 = descripcion
            current_path = {1: last_nivel_1, 2: last_nivel_2, 3: last_nivel_3, 4: last_nivel_4}
        elif nivel == 5:
            last_nivel_5 = descripcion
            current_path = {1: last_nivel_1, 2: last_nivel_2, 3: last_nivel_3, 4: last_nivel_4, 5: last_nivel_5}
        elif nivel == 6:
            last_nivel_6 = descripcion
            current_path = {1: last_nivel_1, 2: last_nivel_2, 3: last_nivel_3, 4: last_nivel_4, 5: last_nivel_5, 6: last_nivel_6}
        elif nivel >= 7:
            current_path[nivel] = descripcion

        # Asignar valores a las columnas de niveles
        for l in range(1, 12):
            if l in current_path:
                df.at[idx, f'Nivel_{l}'] = current_path[l]
            else:
                df.at[idx, f'Nivel_{l}'] = None

    # Renombrar columnas
    nombres_niveles = {
        'Nivel_1': 'Agregado_Total',
        'Nivel_2': 'Etapa_Riesgo',
        'Nivel_3': 'Tipo_Operacion',
        'Nivel_4': 'Clase_Credito',
        'Nivel_5': 'Subclase_Credito',
        'Nivel_6': 'Especificacion_Producto',
        'Nivel_7': 'Calificacion_Riesgo',
        'Nivel_8': 'Piso_Minimo',
        'Nivel_9': 'Subtipo_Operacion',
        'Nivel_10': 'Categoria_Especial'
    }

    df = df.rename(columns=nombres_niveles)

    # Columnas finales
    columnas_finales = [
        'concepto', 'descripcion', 'orden', 'nivel_corregido',
        'Agregado_Total', 'Etapa_Riesgo', 'Tipo_Operacion',
        'Clase_Credito', 'Subclase_Credito', 'Especificacion_Producto',
        'Subtipo_Operacion', 'Calificacion_Riesgo', 'Piso_Minimo', 'Categoria_Especial'
    ]

    columnas_existentes = [col for col in columnas_finales if col in df.columns]

    print("\n🔍 Vista previa de la estructura reordenada:")
    print(f"Total de registros: {len(df)}")
    print(f"Niveles encontrados: {sorted(df['nivel_corregido'].unique())}")

    # Mostrar distribución por niveles
    print("\n📊 Distribución por niveles:")
    nivel_counts = df['nivel_corregido'].value_counts().sort_index()
    for nivel, count in nivel_counts.items():
        print(f"  Nivel {nivel}: {count} registros")

    display(df[columnas_existentes].head(15))

    return df

# Ejecutar la función
if 'df_claves' in globals():
    print("📂 DataFrame 'df_claves' encontrado. Iniciando reestructuración...")
    df_reestructurado = reestructurar_catalogo_colab(df_claves.copy())

    # Validación adicional
    print("\n✅ Validación de la estructura:")
    print("Primeros registros de cada nivel principal:")

    niveles_principales = [1, 2, 3, 4, 5, 6]
    for nivel in niveles_principales:
        subset = df_reestructurado[df_reestructurado['nivel_corregido'] == nivel]
        if not subset.empty:
            print(f"\nNivel {nivel}: {subset.iloc[0]['descripcion']}")
else:
    print("❌ Error: No se encontró el DataFrame 'df_claves'")
    print("Por favor asegúrate de que el DataFrame está cargado correctamente")

📂 DataFrame 'df_claves' encontrado. Iniciando reestructuración...

🔍 Vista previa de la estructura reordenada:
Total de registros: 1660
Niveles encontrados: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

📊 Distribución por niveles:
  Nivel 1: 1 registros
  Nivel 2: 4 registros
  Nivel 3: 6 registros
  Nivel 4: 21 registros
  Nivel 5: 36 registros
  Nivel 6: 133 registros
  Nivel 7: 1305 registros
  Nivel 8: 145 registros
  Nivel 9: 6 registros
  Nivel 10: 3 registros


,concepto,descripcion,orden,nivel_corregido,Agregado_Total,Etapa_Riesgo,Tipo_Operacion,Clase_Credito,Subclase_Credito,Especificacion_Producto,Subtipo_Operacion,Calificacion_Riesgo,Piso_Minimo,Categoria_Especial
0,101800103003,Estimaciones totales (I+II+III+IV),10,1,Estimaciones totales (I+II+III+IV),None,None,None,None,None,None,None,None,None
1,111800304005,I. Cartera base y estimaciones derivadas de la...,20,2,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None
2,101800505014,A). Cartera de crédito,30,3,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,None,None,None,None,None,None,None
3,101801406055,1. Créditos comerciales,40,4,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,None,None,None,None,None,None
4,101808208113,Actividad empresarial o comercial,50,5,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,Actividad empresarial o comercial,None,None,None,None,None
5,101811309053,Operaciones quirografarias,60,6,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,Actividad empresarial o comercial,Operaciones quirografarias,None,None,None,None
6,101805310001,Créditos en cuenta corriente,70,4,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,None,None,None
7,111800111049,Riesgo A-1,80,7,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-1,None,None
8,111800111050,Riesgo A-2,90,7,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-2,None,None
9,111800111051,Riesgo B-1,100,7,Estimaciones totales (I+II+III+IV),I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo B-1,None,None



✅ Validación de la estructura:
Primeros registros de cada nivel principal:

Nivel 1: Estimaciones totales (I+II+III+IV)

Nivel 2: I. Cartera base y estimaciones derivadas de la calificación sobre créditos con riesgo  de crédito etapa 1

Nivel 3: A). Cartera de crédito

Nivel 4: 1. Créditos comerciales

Nivel 5: Actividad empresarial o comercial

Nivel 6: Operaciones quirografarias


Unimos df_principal, df_instituciones y  df_claves segun la clave de institución y concepto respectivamente.

In [152]:
# Unir df_principal con df_instituciones en base a 'institucion' y 'claveinstitucion'
df_resultado = df_principal.merge(df_instituciones,
                                  left_on='institucion',
                                  right_on='claveinstitucion',
                                  how='left')

# Unir df_resultado con df_claves en base a 'concepto'
df_resultado = df_resultado.merge(df_reestructurado,
                                  on='concepto',
                                  how='left')

In [153]:
df_resultado.head(6)

,sector_x,periodo,institucion,moneda,tipo_saldo,orden_presentacion,concepto,importe_pesos,clavesector,sector_y,...,Etapa_Riesgo,Tipo_Operacion,Clase_Credito,Subclase_Credito,Especificacion_Producto,Calificacion_Riesgo,Piso_Minimo,Subtipo_Operacion,Categoria_Especial,Nivel_11
0,40,202501,40002,14,10,10,101800103003,-1.024114e+10,40,BANCA MÚLTIPLE,...,None,None,None,None,None,None,None,None,None,None
1,40,202501,40002,4,10,10,101800103003,-5.785247e+08,40,BANCA MÚLTIPLE,...,None,None,None,None,None,None,None,None,None,None
2,40,202501,40002,14,10,20,111800304005,-4.978340e+09,40,BANCA MÚLTIPLE,...,I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None,None
3,40,202501,40002,4,10,20,111800304005,-3.159043e+08,40,BANCA MÚLTIPLE,...,I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None,None
4,40,202501,40002,14,10,30,101800505014,-4.968993e+09,40,BANCA MÚLTIPLE,...,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,None,None,None,None,None,None,None,None
5,40,202501,40002,4,10,30,101800505014,-3.145321e+08,40,BANCA MÚLTIPLE,...,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,None,None,None,None,None,None,None,None


Empezamos por analizar algunas de las columnas.

In [154]:
df_resultado.value_counts("Calificacion_Riesgo")

,count
Calificacion_Riesgo,
Riesgo E,823527
Riesgo A-1,408778
Riesgo D,407414
Riesgo A-2,407078
Riesgo C-1,406581
Riesgo B-3,406551
Riesgo C-2,406549
Riesgo B-1,406355
Riesgo B-2,406025


In [155]:
df_resultado.columns

Index(['sector_x', 'periodo', 'institucion', 'moneda', 'tipo_saldo',
       'orden_presentacion', 'concepto', 'importe_pesos', 'clavesector',
       'sector_y', 'claveinstitucion', 'nombreinstitucion', 'clavegrupo',
       'nombregrupo', 'descripcion', 'descripcion_identada', 'orden',
       'nivel_corregido', 'Agregado_Total', 'Etapa_Riesgo', 'Tipo_Operacion',
       'Clase_Credito', 'Subclase_Credito', 'Especificacion_Producto',
       'Calificacion_Riesgo', 'Piso_Minimo', 'Subtipo_Operacion',
       'Categoria_Especial', 'Nivel_11'],
      dtype='object')

In [156]:
df_resultado.describe()

,sector_x,periodo,institucion,moneda,tipo_saldo,orden_presentacion,concepto,importe_pesos,clavesector,claveinstitucion,clavegrupo,orden,nivel_corregido
count,4664256.0,4.664256e+06,4.664256e+06,4.664256e+06,4664256.0,4.664256e+06,4.664256e+06,4.664256e+06,4664256.0,4.664256e+06,4.541416e+06,4.664256e+06,4.664256e+06
mean,40.0,2.023132e+05,3.905729e+04,9.052390e+00,10.0,8.350954e+03,1.106115e+11,-1.954334e+07,40.0,3.905729e+04,6.181327e+01,8.350954e+03,6.900038e+00
std,0.0,8.692867e+01,6.422913e+03,4.999726e+00,0.0,4.838274e+03,3.247225e+09,7.759316e+08,0.0,6.422913e+03,1.627351e+00,4.838274e+03,7.413055e-01
min,40.0,2.022010e+05,5.000000e+00,4.000000e+00,10.0,1.000000e+01,1.018001e+11,-2.116276e+11,40.0,5.000000e+00,5.900000e+01,1.000000e+01,1.000000e+00
25%,40.0,2.022100e+05,4.007200e+04,4.000000e+00,10.0,4.120000e+03,1.118021e+11,0.000000e+00,40.0,4.007200e+04,6.100000e+01,4.120000e+03,7.000000e+00
50%,40.0,2.023070e+05,4.013200e+04,1.400000e+01,10.0,8.380000e+03,1.118099e+11,0.000000e+00,40.0,4.013200e+04,6.200000e+01,8.380000e+03,7.000000e+00
75%,40.0,2.024050e+05,4.015000e+04,1.400000e+01,10.0,1.253000e+04,1.118122e+11,0.000000e+00,40.0,4.015000e+04,6.300000e+01,1.253000e+04,7.000000e+00
max,40.0,2.025010e+05,4.016900e+04,1.400000e+01,10.0,1.671000e+04,1.118257e+11,4.405323e+06,40.0,4.016900e+04,6.400000e+01,1.671000e+04,1.000000e+01


In [157]:
df_resultado['sector_x'].value_counts()

,count
sector_x,
40,4664256


In [158]:

df_resultado['clavesector'].value_counts()

,count
clavesector,
40,4664256


In [159]:
df_resultado['tipo_saldo'].value_counts()

,count
tipo_saldo,
10,4664256


Eliminamos las columnas que no tienen datos relevantes.

In [160]:
df_resultado.drop(columns=['sector_x','clavesector','tipo_saldo','descripcion_identada',
    'orden_presentacion',
    'concepto',
    'claveinstitucion',
    'orden','institucion','sector_y','clavegrupo','descripcion','Agregado_Total'], inplace=True)

In [161]:
df_resultado.head(2)

,periodo,moneda,importe_pesos,nombreinstitucion,nombregrupo,nivel_corregido,Etapa_Riesgo,Tipo_Operacion,Clase_Credito,Subclase_Credito,Especificacion_Producto,Calificacion_Riesgo,Piso_Minimo,Subtipo_Operacion,Categoria_Especial,Nivel_11
0,202501,14,-1.024114e+10,Banamex,G-7,1,None,None,None,None,None,None,None,None,None,None
1,202501,4,-5.785247e+08,Banamex,G-7,1,None,None,None,None,None,None,None,None,None,None


In [162]:
df_resultado.columns

Index(['periodo', 'moneda', 'importe_pesos', 'nombreinstitucion',
       'nombregrupo', 'nivel_corregido', 'Etapa_Riesgo', 'Tipo_Operacion',
       'Clase_Credito', 'Subclase_Credito', 'Especificacion_Producto',
       'Calificacion_Riesgo', 'Piso_Minimo', 'Subtipo_Operacion',
       'Categoria_Especial', 'Nivel_11'],
      dtype='object')

En la documentación encontramos que la variable modenda tiene 3 valores que significan:

    2: "Moneda extranjera en dólares", la cual vamos a expresar como USD.
    4: "Moneda extranjera en pesos", la cual vamos a expresar como "ME"
    14: "Moneda nacional, VSM, UMA y UDIS en pesos" la cual vamos a expresar como "MXN".

In [163]:

moneda_dict = {
    2: "USD",
    4: "ME",
    14: "MXN"
}

df_resultado['descripcion_moneda'] = df_resultado['moneda'].map(moneda_dict)

print(df_resultado[['moneda', 'descripcion_moneda']].drop_duplicates())

   moneda descripcion_moneda
0      14                MXN
1       4                 ME


In [164]:
df_resultado.drop(columns=['moneda'], inplace=True)

In [165]:
df_resultado.head(10)

,periodo,importe_pesos,nombreinstitucion,nombregrupo,nivel_corregido,Etapa_Riesgo,Tipo_Operacion,Clase_Credito,Subclase_Credito,Especificacion_Producto,Calificacion_Riesgo,Piso_Minimo,Subtipo_Operacion,Categoria_Especial,Nivel_11,descripcion_moneda
0,202501,-1.024114e+10,Banamex,G-7,1,None,None,None,None,None,None,None,None,None,None,MXN
1,202501,-5.785247e+08,Banamex,G-7,1,None,None,None,None,None,None,None,None,None,None,ME
2,202501,-4.978340e+09,Banamex,G-7,2,I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None,None,MXN
3,202501,-3.159043e+08,Banamex,G-7,2,I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None,None,ME
4,202501,-4.968993e+09,Banamex,G-7,3,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,None,None,None,None,None,None,None,None,MXN
5,202501,-3.145321e+08,Banamex,G-7,3,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,None,None,None,None,None,None,None,None,ME
6,202501,-1.396596e+09,Banamex,G-7,4,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,None,None,None,None,None,None,None,MXN
7,202501,-3.145321e+08,Banamex,G-7,4,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,None,None,None,None,None,None,None,ME
8,202501,-2.832093e+08,Banamex,G-7,5,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,Actividad empresarial o comercial,None,None,None,None,None,None,ME
9,202501,-7.914464e+08,Banamex,G-7,5,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,Actividad empresarial o comercial,None,None,None,None,None,None,MXN


Cambiamos nombres.

In [166]:
# Definir el nuevo orden y nombres de las columnas
nuevo_orden = [
    'periodo',
    'nombreinstitucion',
    'nombregrupo',
    'importe_pesos',
    'descripcion moneda',
    'nivel corregido',
    'Agregado Total',
    'Etapa Riesgo',
    'Tipo Operacion',
    'Clase Credito',
    'Subclase Credito',
    'Especificacion Producto',
    'Subtipo Operacion',
    'Calificacion Riesgo',
    'Piso Minimo',
    'Categoria Especial'
]

# Diccionario para renombrar las columnas
nuevos_nombres = {
    'descripcion_moneda': 'descripcion moneda',
    'nivel_corregido': 'nivel corregido',
    'Agregado_Total': 'Agregado Total',
    'Etapa_Riesgo': 'Etapa Riesgo',
    'Tipo_Operacion': 'Tipo Operacion',
    'Clase_Credito': 'Clase Credito',
    'Subclase_Credito': 'Subclase Credito',
    'Especificacion_Producto': 'Especificacion Producto',
    'Subtipo_Operacion': 'Subtipo Operacion',
    'Calificacion_Riesgo': 'Calificacion Riesgo',
    'Piso_Minimo': 'Piso Minimo',
    'Categoria_Especial': 'Categoria Especial'
}

# Aplicar los cambios
df_final = df_resultado.rename(columns=nuevos_nombres)

# Reordenar las columnas (solo incluyendo las que existan)
columnas_finales = [col for col in nuevo_orden if col in df_final.columns]
df_final = df_final[columnas_finales]

# Mostrar información del resultado
print(f"✅ Estructura final creada con {len(df_final)} registros y {len(df_final.columns)} columnas")
print(f"📊 Columnas finales: {list(df_final.columns)}")
print(f"🔍 Primeras filas:")
display(df_final.head(10))


✅ Estructura final creada con 4664256 registros y 15 columnas
📊 Columnas finales: ['periodo', 'nombreinstitucion', 'nombregrupo', 'importe_pesos', 'descripcion moneda', 'nivel corregido', 'Etapa Riesgo', 'Tipo Operacion', 'Clase Credito', 'Subclase Credito', 'Especificacion Producto', 'Subtipo Operacion', 'Calificacion Riesgo', 'Piso Minimo', 'Categoria Especial']
🔍 Primeras filas:


,periodo,nombreinstitucion,nombregrupo,importe_pesos,descripcion moneda,nivel corregido,Etapa Riesgo,Tipo Operacion,Clase Credito,Subclase Credito,Especificacion Producto,Subtipo Operacion,Calificacion Riesgo,Piso Minimo,Categoria Especial
0,202501,Banamex,G-7,-1.024114e+10,MXN,1,None,None,None,None,None,None,None,None,None
1,202501,Banamex,G-7,-5.785247e+08,ME,1,None,None,None,None,None,None,None,None,None
2,202501,Banamex,G-7,-4.978340e+09,MXN,2,I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None
3,202501,Banamex,G-7,-3.159043e+08,ME,2,I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None
4,202501,Banamex,G-7,-4.968993e+09,MXN,3,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,None,None,None,None,None,None,None
5,202501,Banamex,G-7,-3.145321e+08,ME,3,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,None,None,None,None,None,None,None
6,202501,Banamex,G-7,-1.396596e+09,MXN,4,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,None,None,None,None,None,None
7,202501,Banamex,G-7,-3.145321e+08,ME,4,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,None,None,None,None,None,None
8,202501,Banamex,G-7,-2.832093e+08,ME,5,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,Actividad empresarial o comercial,None,None,None,None,None
9,202501,Banamex,G-7,-7.914464e+08,MXN,5,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,1. Créditos comerciales,Actividad empresarial o comercial,None,None,None,None,None


In [167]:
df_final.head(5)

,periodo,nombreinstitucion,nombregrupo,importe_pesos,descripcion moneda,nivel corregido,Etapa Riesgo,Tipo Operacion,Clase Credito,Subclase Credito,Especificacion Producto,Subtipo Operacion,Calificacion Riesgo,Piso Minimo,Categoria Especial
0,202501,Banamex,G-7,-1.024114e+10,MXN,1,None,None,None,None,None,None,None,None,None
1,202501,Banamex,G-7,-5.785247e+08,ME,1,None,None,None,None,None,None,None,None,None
2,202501,Banamex,G-7,-4.978340e+09,MXN,2,I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None
3,202501,Banamex,G-7,-3.159043e+08,ME,2,I. Cartera base y estimaciones derivadas de la...,None,None,None,None,None,None,None,None
4,202501,Banamex,G-7,-4.968993e+09,MXN,3,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,None,None,None,None,None,None,None


In [168]:
df_final.columns

Index(['periodo', 'nombreinstitucion', 'nombregrupo', 'importe_pesos',
       'descripcion moneda', 'nivel corregido', 'Etapa Riesgo',
       'Tipo Operacion', 'Clase Credito', 'Subclase Credito',
       'Especificacion Producto', 'Subtipo Operacion', 'Calificacion Riesgo',
       'Piso Minimo', 'Categoria Especial'],
      dtype='object')

In [169]:
df_final.value_counts('Calificacion Riesgo')

,count
Calificacion Riesgo,
Riesgo E,823527
Riesgo A-1,408778
Riesgo D,407414
Riesgo A-2,407078
Riesgo C-1,406581
Riesgo B-3,406551
Riesgo C-2,406549
Riesgo B-1,406355
Riesgo B-2,406025


In [170]:
df_final.value_counts('Piso Minimo')

,count
Piso Minimo,
Piso MI,492848


Eliminamos los datos que no son creditos aprobados.
Estos al menos deben tener nivel 7, es decir, una calificación.

In [171]:
df_final = df_final[df_final['nivel corregido'] >= 7]

In [172]:
df_final.head(10)

,periodo,nombreinstitucion,nombregrupo,importe_pesos,descripcion moneda,nivel corregido,Etapa Riesgo,Tipo Operacion,Clase Credito,Subclase Credito,Especificacion Producto,Subtipo Operacion,Calificacion Riesgo,Piso Minimo,Categoria Especial
14,202501,Banamex,G-7,-2.250218e+08,MXN,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-1,None,None
15,202501,Banamex,G-7,-2.468053e+07,ME,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-1,None,None
16,202501,Banamex,G-7,-1.153210e+08,MXN,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-2,None,None
17,202501,Banamex,G-7,-9.164558e+06,ME,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-2,None,None
18,202501,Banamex,G-7,0.000000e+00,ME,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo B-1,None,None
19,202501,Banamex,G-7,-1.242725e+07,MXN,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo B-1,None,None
20,202501,Banamex,G-7,-7.484578e+06,ME,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo B-2,None,None
21,202501,Banamex,G-7,-6.449692e+06,MXN,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo B-2,None,None
22,202501,Banamex,G-7,0.000000e+00,ME,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo B-3,None,None
23,202501,Banamex,G-7,-1.797418e+07,MXN,7,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo B-3,None,None


In [173]:
df_final.drop(columns=['nivel corregido'], inplace=True)

In [174]:
df_final.head(5)

,periodo,nombreinstitucion,nombregrupo,importe_pesos,descripcion moneda,Etapa Riesgo,Tipo Operacion,Clase Credito,Subclase Credito,Especificacion Producto,Subtipo Operacion,Calificacion Riesgo,Piso Minimo,Categoria Especial
14,202501,Banamex,G-7,-2.250218e+08,MXN,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-1,None,None
15,202501,Banamex,G-7,-2.468053e+07,ME,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-1,None,None
16,202501,Banamex,G-7,-1.153210e+08,MXN,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-2,None,None
17,202501,Banamex,G-7,-9.164558e+06,ME,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo A-2,None,None
18,202501,Banamex,G-7,0.000000e+00,ME,I. Cartera base y estimaciones derivadas de la...,A). Cartera de crédito,Créditos en cuenta corriente,None,None,None,Riesgo B-1,None,None


In [175]:
df_final.describe()

,periodo,importe_pesos
count,4.087178e+06,4.087178e+06
mean,2.023132e+05,-3.299056e+06
std,8.692785e+01,8.266926e+07
min,2.022010e+05,-1.257387e+10
25%,2.022100e+05,0.000000e+00
50%,2.023070e+05,0.000000e+00
75%,2.024050e+05,0.000000e+00
max,2.025010e+05,4.405323e+06


In [176]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4087178 entries, 14 to 4664241
Data columns (total 14 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   periodo                  int64  
 1   nombreinstitucion        object 
 2   nombregrupo              object 
 3   importe_pesos            float64
 4   descripcion moneda       object 
 5   Etapa Riesgo             object 
 6   Tipo Operacion           object 
 7   Clase Credito            object 
 8   Subclase Credito         object 
 9   Especificacion Producto  object 
 10  Subtipo Operacion        object 
 11  Calificacion Riesgo      object 
 12  Piso Minimo              object 
 13  Categoria Especial       object 
dtypes: float64(1), int64(1), object(12)
memory usage: 467.7+ MB


In [ ]:
df_final['periodo'] = df_final['periodo'].astype(str)

# Crear nuevas columnas para el año y el mes
df_final['Año'] = df_final['periodo'].str[:4]  # Obtener los primeros 4 caracteres para el año
df_final['Mes'] = df_final['periodo'].str[4:6]  # Obtener los últimos 2 caracteres para el mes


In [ ]:
df_final.head(10)

In [ ]:
df_final.info()

In [ ]:
df_final['Año'] = df_final['Año'].astype(int)
df_final['Mes'] = df_final['Mes'].astype(int)

In [ ]:
df_final.drop(columns = ['periodo'], inplace=True)

In [ ]:
df_final.info()

In [ ]:
df_final.head(4)

In [ ]:
# Definir el nuevo orden y nombres de las columnas
nuevo_orden = [
    'Año',
    'Mes',
    'Nombre Institución',
    'Nombre Grupo',
    'Importe Pesos',
    'Descripción Moneda',
    'Agregado Total',
    'Etapa Riesgo',
    'Tipo Operacion',
    'Clase Credito',
    'Subclase Credito',
    'Especificacion Producto',
    'Subtipo Operacion',
    'Calificacion Riesgo',
    'Piso Minimo',
    'Categoria Especial'
]

# Diccionario para renombrar las columnas
nuevos_nombres = {
    'nombreinstitucion': 'Nombre Institución',
    'nombregrupo': 'Nombre Grupo',
    'importe_pesos': 'Importe Pesos',
    'descripcion moneda': 'Descripción Moneda',
    'Agregado_Total': 'Agregado Total',
    'Etapa_Riesgo': 'Etapa Riesgo',
    'Tipo_Operacion': 'Tipo Operacion',
    'Clase_Credito': 'Clase Credito',
    'Subclase_Credito': 'Subclase Credito',
    'Especificacion_Producto': 'Especificacion Producto',
    'Subtipo_Operacion': 'Subtipo Operacion',
    'Calificacion_Riesgo': 'Calificacion Riesgo',
    'Piso_Minimo': 'Piso Minimo',
    'Categoria_Especial': 'Categoria Especial'
}

# Aplicar los cambios
df_final = df_final.rename(columns=nuevos_nombres)


In [ ]:
df_final.head(3)

In [ ]:
# Reordenar las columnas (solo incluyendo las que existan)
columnas_finales = [col for col in nuevo_orden if col in df_final.columns]
df_final = df_final[columnas_finales]

In [ ]:
df_final.head(5)

Eliminamos los elementos con importe 0.

In [ ]:

df_final = df_final[df_final['Importe Pesos'] != 0]


Volvemos todos los importes positivos, para hacer más fácil su manejo y graficar sin dificultad.

In [ ]:
df_final['Importe Pesos'] = df_final['Importe Pesos'].abs()


In [ ]:
df_final = df_final[df_final['Nombre Institución'] != 'Total Banca Múltiple']

Ahora que nuestra base de datos esta limpia, comezamos a hacer estaditica descrptiva.

Nuestra base de datos comprende desde Enero de 2022 a Enero de 2025.

¿Cuántos créditos se aprobaron durante ese periodo?

In [ ]:
num = df_final['Calificacion Riesgo'].count()
print(f"El número de créditos aprobados es: {num}")

In [ ]:
# prompt: suma el importe total

# Calculate the sum of 'Importe Pesos'
total_importe = df_final['Importe Pesos'].sum()

# Print the total importe
print(f"El importe total de los créditos aprobados es: {total_importe}")


Importe total de créditos por calificación de riesgo

In [ ]:
df_final.groupby('Calificacion Riesgo')['Importe Pesos'].sum().sort_values(ascending=False)

Importe promedio por institución o grupo

In [ ]:

df_final.groupby('Nombre Institución')['Importe Pesos'].mean()

In [ ]:
df_final.groupby('Nombre Grupo')['Importe Pesos'].sum()

En que mes se presta mas dinero en creditos?

In [ ]:

df_aprobados = df_final[df_final['Importe Pesos'] > 0]
creditos_por_mes = df_aprobados.groupby('Mes')['Importe Pesos'].sum()
mes_maximo = creditos_por_mes.idxmax()
importe_maximo = creditos_por_mes.max()
print(f"El mes con más créditos aprobados es el mes {mes_maximo} con un total de {importe_maximo} pesos.")

¿Cómo se distribuyen los créditos aprobados entre las Instituciones?

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
banco_counts = df_final['Nombre Institución'].value_counts()
top_15_bancos = banco_counts.head(15)
otros_bancos = banco_counts.tail(banco_counts.shape[0] - 15).sum()
top_bancos = pd.concat([top_15_bancos, pd.Series({'Otros': otros_bancos})])
plt.figure(figsize=(10, 7))
top_bancos.plot(kind='pie', autopct='%1.1f%%', startangle=90, colors=plt.cm.Paired.colors)
plt.title('Distribución de Créditos por Banco')
plt.ylabel('')  # No mostrar etiqueta en el eje Y
plt.show()

In [ ]:
df_final.value_counts('Calificacion Riesgo')

In [ ]:
# prompt: agrupa los datos por año, mes y calificacion de riesgo y cuenta cada uno

# Group the data by year, month, and risk rating, then count occurrences
grouped_data = df_final.groupby(['Año', 'Mes', 'Calificacion Riesgo']).size().reset_index(name='Count')

grouped_data


In [ ]:
grouped_data.tail(12)

In [ ]:
banco_importe = df_final.groupby('Nombre Institución')['Importe Pesos'].sum()
banco_importe = banco_importe.abs()
top_10_bancos_importe = banco_importe.sort_values(ascending=False).head(10)
otros_bancos_importe = banco_importe.tail(banco_importe.shape[0] - 10).sum()

# 5. Creamos una nueva serie con los top 10 y el total de los demás bancos
top_bancos_importe = pd.concat([top_10_bancos_importe, pd.Series({'Otros': otros_bancos_importe})])

# 6. Creamos el gráfico de pastel con el importe en pesos (ahora en valores absolutos)
plt.figure(figsize=(10, 7))
top_bancos_importe.plot(kind='pie', autopct='%1.1f%%', startangle=90, colors=plt.cm.Paired.colors)

# Título y personalización
plt.title('Distribución de Dinero Prestado por Banco (Importe en Pesos Absoluto)')
plt.ylabel('')  # No mostrar etiqueta en el eje Y
plt.show()

In [ ]:
plt.figure(figsize=(70,15))
df_final.groupby(['Año', 'Mes', 'Calificacion Riesgo'])['Importe Pesos'].sum().unstack().plot()
plt.show()

In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_final,
    x='Mes',
    y='Importe Pesos',
    hue='Año',
    marker='o',
    errorbar=None
)

plt.title('Evolución de Importes por Mes y Año')
plt.xlabel('Mes')
plt.ylabel('Importe Pesos')
plt.grid(True)
plt.legend(title='Año')
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
plt.xticks(range(1, 13), meses)

plt.show()

Evolución del Importe Total por Mes/Año para cada Institución/Grupo

In [ ]:
top_bancos = df_final.groupby('Nombre Institución')['Importe Pesos'].sum().abs().nlargest(10).index

# Filtrar y preparar datos
df_top = df_final[df_final['Nombre Institución'].isin(top_bancos)].copy()
df_top['Fecha'] = pd.to_datetime(df_top['Año'].astype(str) + '-' + df_top['Mes'].astype(str))
fig = px.line(
    df_top.groupby(['Fecha', 'Nombre Institución'], as_index=False)['Importe Pesos'].sum(),
    x='Fecha',
    y='Importe Pesos',
    color='Nombre Institución',
    title='Evolución Mensual - Top 10 Bancos (por volumen absoluto)',
    labels={'Importe Pesos': 'Importe (millones)', 'Fecha': 'Periodo'},
    height=600
)

fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.update_layout(hovermode='x unified')
fig.show()

In [ ]:
etiquetas = df_final['Calificacion Riesgo'].value_counts().index
valor = df_final['Calificacion Riesgo'].value_counts().values

df_riesgo = pd.DataFrame({
    'Calificacion': etiquetas,
    'Cantidad': valor
})


orden_riesgo = ['Riesgo A-1', 'Riesgo A-2', 'Riesgo B-1', 'Riesgo B-2', 'Riesgo B-3','Riesgo C-1'
               'Riesgo C-2', 'Riesgo D', 'Riesgo E']
df_riesgo['Calificacion'] = pd.Categorical(df_riesgo['Calificacion'], categories=orden_riesgo, ordered=True)
df_riesgo = df_riesgo.sort_values('Calificacion')


fig = px.bar(df_riesgo,
             x='Calificacion',
             y='Cantidad',
             color='Calificacion',
             color_discrete_sequence=px.colors.sequential.Greens_r,
             title='Distribución de Calificaciones de Riesgo en la Cartera',
             labels={'Cantidad': 'Número de Créditos', 'Calificacion': 'Calificación de Riesgo'},
             text='Cantidad')

# Personalizar el gráfico
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(
    xaxis_title="Calificación de Riesgo",
    yaxis_title="Cantidad de Créditos",
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis={'type': 'category'},
    hovermode="x unified"
)
fig.show()

In [ ]:
top_15_bancos = df_final['Nombre Institución'].value_counts().nlargest(15).index

df_top15 = df_final[df_final['Nombre Institución'].isin(top_15_bancos)].copy()

df_riesgo_top15 = df_top15.groupby(
    ['Nombre Institución', 'Calificacion Riesgo'],
    as_index=False
)['Importe Pesos'].sum()

pivot_top15 = df_riesgo_top15.pivot_table(
    index='Nombre Institución',
    columns='Calificacion Riesgo',
    values='Importe Pesos'
).fillna(0)

orden_riesgo = ['Riesgo A-1', 'Riesgo A-2', 'Riesgo B-1', 'Riesgo B-2', 'Riesgo B-3', 'Riesgo C-1',
               'Riesgo C-2', 'Riesgo D', 'Riesgo E']

categorias_existentes = [c for c in orden_riesgo if c in pivot_top15.columns]
pivot_percent = pivot_top15[categorias_existentes].div(pivot_top15[categorias_existentes].sum(axis=1), axis=0) * 100

plt.figure(figsize=(20, 10))
sns.heatmap(
    pivot_percent,
    annot=True,
    fmt='.1f',
    cmap='Greens',
    linewidths=0.5,
    linecolor='lightgray',
    cbar_kws={'label': 'Porcentaje del Total por Banco', 'shrink': 0.8},
    annot_kws={'size': 9}
)

plt.title('Distribución Porcentual de Importes por Calificación de Riesgo\nTop 15 Bancos por Cantidad de Créditos',
          fontsize=16, pad=20)
plt.xlabel('Calificación de Riesgo', fontsize=16)
plt.ylabel('Institución Financiera', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=16)

ax = plt.gca()
ax.set_yticklabels(ax.get_yticklabels(), va='center')
plt.tight_layout()
plt.show()

In [ ]:
df_final.columns


In [ ]:
# Contar cuántos créditos hay por banco y calificación de riesgo
df_riesgo = df_final.groupby(['Nombre Institución', 'Calificacion Riesgo']).size().reset_index(name='Cantidad Créditos')

df_pivot = df_riesgo.pivot_table(index='Nombre Institución',
                                 columns='Calificacion Riesgo',
                                 values='Cantidad Créditos',
                                 fill_value=0)


df_pivot['Total Créditos Banco'] = df_pivot.sum(axis=1)

df_banco = df_pivot.sort_values('Total Créditos Banco', ascending=False).reset_index()

In [ ]:
df_riesgo.head(3)

In [ ]:

df_banco = df_riesgo.pivot_table(
    index='Nombre Institución',
    columns='Calificacion Riesgo',
    values='Cantidad Créditos',
    fill_value=0
)

df_banco['Total Créditos Banco'] = df_banco.sum(axis=1)


df_banco = df_banco.reset_index()

In [ ]:
df_banco.head(3)

In [ ]:

for col in df_banco.columns:
    if col != 'Nombre Institución' and col != 'Total Créditos Banco':
        df_banco[col + ' (%)'] = (df_banco[col] / df_banco['Total Créditos Banco']) * 100

df_banco.head()


In [ ]:
percentage_cols = [col for col in df_banco.columns if '(' in col]
df_banco = df_banco[['Nombre Institución'] + percentage_cols]
df_banco.head()


In [ ]:
def clasificar_banco(row):
    bajo_riesgo = row['Riesgo A-1 (%)'] + row['Riesgo A-2 (%)']
    riesgo_alto = (
        row['Riesgo C-1 (%)'] +
        row['Riesgo C-2 (%)'] +
        row['Riesgo D (%)'] +
        row['Riesgo E (%)']
    )

    if bajo_riesgo >= 60:
        return 'Bajo riesgo'
    elif riesgo_alto >= 40:
        return 'Alto riesgo'
    else:
        return 'Riesgo moderado'

df_banco['Clasificación'] = df_banco.apply(clasificar_banco, axis=1)

In [ ]:
df_banco.head(3)

In [ ]:

df_banco_sorted = df_banco.sort_values('Clasificación')

plt.figure(figsize=(10, 6))


sns.barplot(
    data=df_banco_sorted,
    y='Nombre Institución',
    x='Riesgo A-1 (%)',
    hue='Clasificación',
    dodge=False,
    palette={'Bajo riesgo': 'green', 'Riesgo moderado': 'orange', 'Alto riesgo': 'red'}
)


plt.xlabel('Proporción en Riesgo A-1 (%)')
plt.ylabel('Banco')
plt.title('Clasificación de riesgo por banco')
plt.legend(title='Clasificación')
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Copia del DataFrame
df = df_final.copy()

# Columnas irrelevantes
cols_a_eliminar = ['Piso Minimo','Categoria Especial','Especificacion Producto','Subtipo Operacion']
df.drop(columns=cols_a_eliminar, inplace=True, errors='ignore')

# Valores faltantes
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Desconocido')
for col in df.select_dtypes(exclude='object').columns:
    df[col] = df[col].fillna(0)

# Codificar Riesgo jerárquico
df['Riesgo_simple'] = df['Calificacion Riesgo'].str.extract(r'([A-E])', expand=False)
mapa_riesgo = {'A':0,'B':1,'C':2,'D':3,'E':4}
df['Calificacion Riesgo'] = df['Riesgo_simple'].map(mapa_riesgo)
df.drop(columns=['Riesgo_simple'], inplace=True)

# Lista de columnas a codificar numéricamente
cols_codificar = ['Nombre Institución', 'Tipo Operacion', 'Clase Credito', 'Subclase Credito',
                  'Etapa Riesgo', 'Descripción Moneda']

codificadores = {}
for col in cols_codificar:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    codificadores[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"Mapa {col}: {codificadores[col]}")

# Escalar columnas numéricas
scaler = StandardScaler()
num_cols = ['Año', 'Mes', 'Importe Pesos']
df[num_cols] = scaler.fit_transform(df[num_cols])

print("Dimensiones finales:", df.shape)
df.head()


In [ ]:
df = df.drop(columns=['Nombre Grupo'])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# --- Librerías básicas ---
import pandas as pd
import numpy as np
import itertools # Import the itertools library

# --- Preprocesamiento ---
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# --- Modelos de clasificación ---
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# --- Clustering ---
from sklearn.cluster import KMeans

# --- Métricas ---
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración gráfica
sns.set(style='whitegrid')

# Function to plot confusion matrix
def plot_confusion_matrix(cm, classes, title='Confusion matrix', cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if False else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

In [ ]:
X = df.drop(columns=['Calificacion Riesgo'])
y = df['Calificacion Riesgo']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.02, random_state=42)

LR

In [ ]:
modelo_LR = LogisticRegression(max_iter = 1000).fit(X_train, y_train)
modelo_LR

In [ ]:
pred = modelo_LR.predict(X_test)
pred

In [ ]:
lr_accuracy = accuracy_score(y_test, pred)
lr_accuracy


In [ ]:
lr_precision = precision_score(y_test, pred, average='macro')
lr_precision


In [ ]:
lr_recall = recall_score(y_test, pred, average='macro')
lr_recall

In [ ]:
lr_f1 = f1_score(y_test, pred, average='macro')
lr_f1

In [ ]:
lr_report = classification_report(y_test, pred)
print(lr_report)

In [ ]:
lr_cm = confusion_matrix(y_test, pred)
plot_confusion_matrix(lr_cm, ['A', 'B', 'c', 'D', 'E'])

In [ ]:
lr_cm

In [ ]:
TPrate = lr_cm[1,1] / (lr_cm[1,0] + lr_cm[1,1])
TPrate

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Binarizamos las clases (one-hot)
y_bin = label_binarize(y_test, classes=[0,1,2,3,4])
n_classes = y_bin.shape[1]

# Obtener probabilidades del modelo
y_score = modelo_LR.predict_proba(X_test)

# Graficar ROC por clase
plt.figure(figsize=(8,6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Clase {i} (AUC = {roc_auc:.2f})')

plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC multiclase (one-vs-rest)')
plt.legend()
plt.show()


KNN

In [ ]:
Modelo_KNN=KNeighborsClassifier().fit(X_train,y_train)
Modelo_KNN

In [ ]:
pred_KNN=Modelo_KNN.predict(X_test)
pred_KNN

In [ ]:
knn_accuracy = accuracy_score(y_test, pred_KNN)
knn_accuracy

In [ ]:
knn_precision = precision_score(y_test, pred_KNN, average='macro')
knn_precision

In [ ]:
knn_recall = recall_score(y_test, pred_KNN, average='macro')
knn_recall


In [ ]:
knn_f1 = f1_score(y_test, pred_KNN, average='macro')
knn_f1

In [ ]:
knn_report = classification_report(y_test, pred_KNN)
print(knn_report)

In [ ]:
knn_cm = confusion_matrix(y_test, pred_KNN)
plot_confusion_matrix(knn_cm, ['A', "B","C",'D', 'E'])

In [ ]:
knn_cm

In [ ]:
TPrate = knn_cm[1,1] / (knn_cm[1,0] + knn_cm[1,1])
TPrate

In [ ]:
# Binarizamos las clases (one-hot)
y_bin = label_binarize(y_test, classes=[0,1,2,3,4])
n_classes = y_bin.shape[1]

# Obtener probabilidades del modelo
y_score = modelo_LR.predict_proba(X_test)

# Graficar ROC por clase
plt.figure(figsize=(8,6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Clase {i} (AUC = {roc_auc:.2f})')

plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC multiclase (one-vs-rest)')
plt.legend()
plt.show()


Decision Tree

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

In [ ]:

# --- Predicciones ---
pred_dt = dt.predict(X_test)

# --- Métricas de desempeño ---
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

dt_accuracy = accuracy_score(y_test, pred_dt)
dt_precision = precision_score(y_test, pred_dt, average='macro')
dt_recall = recall_score(y_test, pred_dt, average='macro')
dt_f1 = f1_score(y_test, pred_dt, average='macro')

print("Accuracy:", dt_accuracy)
print("Precision (macro):", dt_precision)
print("Recall (macro):", dt_recall)
print("F1-score (macro):", dt_f1)

# Reporte completo
print("\nClassification Report:\n")
print(classification_report(y_test, pred_dt))

# Matriz de confusión
dt_cm = confusion_matrix(y_test, pred_dt)
plt.figure(figsize=(8,6))
sns.heatmap(dt_cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión - Árbol de Decisión')
plt.show()

# --- Curva ROC multiclase ---
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Binarizamos las clases
y_bin = label_binarize(y_test, classes=[0,1,2,3,4])
n_classes = y_bin.shape[1]

# Obtener probabilidades del modelo
y_score = dt.predict_proba(X_test)

# Graficar ROC por clase (one-vs-rest)
plt.figure(figsize=(8,6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Clase {i} (AUC = {roc_auc:.2f})')

plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC multiclase - Árbol de Decisión (one-vs-rest)')
plt.legend()
plt.show()


Random Forest

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

In [ ]:
pred_rf = rf.predict(X_test)

# --- Métricas de desempeño ---
rf_accuracy = accuracy_score(y_test, pred_rf)
rf_precision = precision_score(y_test, pred_rf, average='macro')
rf_recall = recall_score(y_test, pred_rf, average='macro')
rf_f1 = f1_score(y_test, pred_rf, average='macro')

print("Accuracy:", rf_accuracy)
print("Precision (macro):", rf_precision)
print("Recall (macro):", rf_recall)
print("F1-score (macro):", rf_f1)

# Reporte completo
print("\nClassification Report:\n")
print(classification_report(y_test, pred_rf))

# Matriz de confusión
rf_cm = confusion_matrix(y_test, pred_rf)
plt.figure(figsize=(8,6))
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión - Random Forest')
plt.show()

# --- Curva ROC multiclase ---
# Binarizamos las clases
y_bin = label_binarize(y_test, classes=[0,1,2,3,4])
n_classes = y_bin.shape[1]

# Obtener probabilidades del modelo
y_score = rf.predict_proba(X_test)

# Graficar ROC por clase (one-vs-rest)
plt.figure(figsize=(8,6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Clase {i} (AUC = {roc_auc:.2f})')

plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC multiclase - Random Forest (one-vs-rest)')
plt.legend()
plt.show()

In [ ]:
models_data = {
    'Model': ['Logistic Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest'],
    'Recall': [lr_recall, knn_recall, dt_recall, rf_recall],
}

df_models = pd.DataFrame(models_data)
display(df_models)

In [ ]:
fig = px.bar(df_models, x='Model', y='Recall', color='Model', title='Recall Comparacion')
fig.show()

In [ ]:
models_data = {
    'Model': ['Logistic Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest'],
    'Accuracy': [lr_accuracy, knn_accuracy, dt_accuracy, rf_accuracy],
}

df_models = pd.DataFrame(models_data)
display(df_models)

In [ ]:
fig = px.bar(df_models, x='Model', y='Accuracy', color='Model', title='Accuracy Comparacion')
fig.show()

In [ ]:
models_data = {
    'Model': ['Logistic Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest'],
    'Precision': [lr_precision, knn_precision, dt_precision, rf_precision],
}

df_models = pd.DataFrame(models_data)
display(df_models)

In [ ]:
fig = px.bar(df_models, x='Model', y='Precision', color='Model', title='Precision Comparacion')
fig.show()

In [ ]:
models_data = {
    'Model': ['Logistic Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest'],
    'R1': [lr_f1, knn_f1, dt_f1, rf_f1],
}

df_models = pd.DataFrame(models_data)
display(df_models)

In [ ]:
fig = px.bar(df_models, x='Model', y='R1', color='Model', title='R1')
fig.show()

In [ ]:
df_final["Calificacion Riesgo"].value_counts()